In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy

sys.path.append(os.path.abspath("../"))

In [ ]:
def read_csv_file(file_path):
    """
    Reads a CSV file from the given path and returns a pandas DataFrame.

    Parameters:
    file_path (str): Path to the CSV file.

    Returns:
    pd.DataFrame: DataFrame containing the CSV data.
    """
    try:
        df = pd.read_csv(file_path)
        return df
    except FileNotFoundError:
        print(f"Error: The file at {file_path} was not found.")
        return None
    except Exception as e:
        print(f"An error occurred: {e}")
        return None


baseline_run_1 = read_csv_file("../../data/evaluation/classifier-basic/run_1.csv")
baseline_run_2 = read_csv_file("../../data/evaluation/classifier-basic/run_2.csv")
baseline_run_3 = read_csv_file("../../data/evaluation/classifier-basic/run_3.csv")

run_1 = read_csv_file("../../data/evaluation/classifier-saturation/run_1.csv")
run_2 = read_csv_file("../../data/evaluation/classifier-saturation/run_2.csv")
run_3 = read_csv_file("../../data/evaluation/classifier-saturation/run_3.csv")

In [ ]:
n_context = [8, 16, 32, 64, 128]
runs = [run_1, run_2, run_3]

cla_runs = {}

for _, n_ctx in enumerate(n_context):
    for j, run in enumerate(runs):
        cla_runs[f"n_ctx_{n_ctx}_run{j}"] = run[run["n_context"] == n_ctx]

In [ ]:
baseline_runs = [baseline_run_1, baseline_run_2, baseline_run_3]


for j, run in enumerate(baseline_runs):
    cla_runs[f"n_ctx_0_run{j}"] = run[run["architecture"] == "conv_v4"]

In [ ]:
# ── Data ──────────────────────────────────────────────────────────────────────
ctx_sizes = ["0", "8", "16", "32", "64", "128"]

classes = ["Ball", "Elfmeterpunkt", "Linienkreuzung"]

In [ ]:
ceilings = {
    "Ball": float(cla_runs["n_ctx_0_run0"]["encoder_recall_at_k_ball"].iloc[0]),
    "Elfmeterpunkt": float(cla_runs["n_ctx_0_run0"]["encoder_recall_at_k_penaltyMark"].iloc[0]),
    "Linienkreuzung": float(cla_runs["n_ctx_0_run0"]["encoder_recall_at_k_intersections"].iloc[0]),
}

In [ ]:
ap_values = {
    "Ball": [
        [
            float(cla_runs[f"n_ctx_{ctx}_run{run}"]["ball_ap_pooled"].iloc[0])
            for run in range(len(runs))
        ]
        for ctx in [0] + n_context
    ],
    "Elfmeterpunkt": [
        [
            float(cla_runs[f"n_ctx_{ctx}_run{run}"]["penaltyMark_ap_pooled"].iloc[0])
            for run in range(len(runs))
        ]
        for ctx in [0] + n_context
    ],
    "Linienkreuzung": [
        [
            float(cla_runs[f"n_ctx_{ctx}_run{run}"]["intersections_mAP"].iloc[0])
            for run in range(len(runs))
        ]
        for ctx in [0] + n_context
    ],
}

euc_values = {
    "Ball": [
        [
            float(cla_runs[f"n_ctx_{ctx}_run{run}"]["classifier_euc_error_ball"].iloc[0])
            for run in range(len(runs))
        ]
        for ctx in [0] + n_context
    ],
    "Elfmeterpunkt": [
        [
            float(cla_runs[f"n_ctx_{ctx}_run{run}"]["classifier_euc_error_penaltyMark"].iloc[0])
            for run in range(len(runs))
        ]
        for ctx in [0] + n_context
    ],
    "Linienkreuzung": [
        [
            float(cla_runs[f"n_ctx_{ctx}_run{run}"]["classifier_euc_error_intersections"].iloc[0])
            for run in range(len(runs))
        ]
        for ctx in [0] + n_context
    ],
}


In [ ]:
# ── Shared Layoutparameters ─────────────────────────────────────────────────
x = np.arange(len(ctx_sizes))
n_classes = len(classes)
bar_width = 0.22
offsets = np.linspace(-(n_classes - 1) / 2, (n_classes - 1) / 2, n_classes) * bar_width
colors = ["#0072B2", "#E69F00", "#009E73"]  # Okabe-Ito
mean_color = "#CC79A7"


# ── Helper: Ceilings ─────────────────────────────────────────────
def draw_ceilings(ax, ceiling_dict):
    for cls, color in zip(classes, colors, strict=True):
        ceiling = ceiling_dict[cls]
        ax.axhline(
            y=ceiling,
            color=color,
            linewidth=1.2,
            linestyle="--",
            alpha=0.8,
            zorder=2,
        )


# ── Helper: Styles of axes ───────────────────────────────────────────────
def style_ax(ax):
    ax.set_xticks(x)
    ax.set_xticklabels(ctx_sizes, fontsize=11)
    ax.set_xlabel(r"$n_\mathrm{ctx}$", fontsize=11)
    ax.yaxis.grid(True, linestyle="--", alpha=0.6, zorder=0)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)


# ── Helper: Points + Mean + Connecting Lines ──────────────────────────────
def draw_points_and_mean(ax, data_dict):
    for cls, color, offset in zip(classes, colors, offsets, strict=True):
        means = []
        for i, samples in enumerate(data_dict[cls]):
            samples = np.array(samples)
            mean = np.mean(samples)
            stdd = np.std(samples)
            means.append(mean)

            # Raw datapoint: random horizontal alignment (Jitter)
            jitter = np.linspace(-0.06, 0.06, len(samples))
            ax.scatter(
                x[i] + offset + jitter,
                samples,
                color=color,
                s=30,
                zorder=5,
                alpha=0.7,
                edgecolors="white",
                linewidths=0.4,
            )
            print("Arch", i, cls, " Mean ", mean * 100, "Stdd ", stdd * 100)
            # Mean
            ax.hlines(
                mean,
                x[i] + offset - 0.09,
                x[i] + offset + 0.09,
                colors=color,
                linewidth=2.5,
                zorder=6,
            )

        # Connect the means for this class
        ax.plot(
            x + offset,
            means,
            color=color,
            linewidth=0.7,
            linestyle="-",
            zorder=4,
            alpha=0.9,
            marker="",
            markersize=4,
            markerfacecolor="white",
            markeredgecolor=color,
            markeredgewidth=1.2,
        )


# ── Helper: Legend ───────────────────────────
def add_legend(ax, with_ceiling=True):
    handles = [
        plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=color, markersize=7, label=cls)
        for cls, color in zip(classes, colors, strict=True)
    ]

    if with_ceiling:
        ceiling_handles = [
            plt.Line2D(
                [0],
                [0],
                color=color,
                linewidth=1.2,
                linestyle="--",
                alpha=0.8,
                label=r"Recall$@K_c$ für " + f"{cls}",
            )
            for cls, color in zip(classes, colors, strict=True)
        ]
        handles += ceiling_handles

    ax.legend(
        handles=handles,
        loc="lower right",
        framealpha=0.9,
        fontsize=9,
        ncols=2 if with_ceiling else 1,
    )


# — Plot 1: AP / mAP ————————————————————————————————————————————————————————————
fig, ax = plt.subplots(figsize=(9, 5))
draw_points_and_mean(ax, ap_values)
ax.set_ylim(0.4, 1.05)
draw_ceilings(ax, ceilings)
style_ax(ax)
ax.set_ylabel("AP / mAP", fontsize=11)
add_legend(ax)
plt.tight_layout()
plt.savefig("../../plots/ctx_saturation/average_precision.pdf")

# — Plot 2: Euclidean Error ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
ymax = max(v for vals in euc_values.values() for samples in vals for v in samples) * 1.2
draw_points_and_mean(ax, euc_values)
ax.set_ylim(0, 3.5)
style_ax(ax)
ax.set_ylabel(r"$\bar{e}^\prime$ [px]", fontsize=11)
add_legend(ax, with_ceiling=False)
plt.tight_layout()
plt.savefig("../../plots/ctx_saturation/euclidean_error.pdf")

plt.tight_layout()
plt.show()

In [ ]:
for cls in classes:
    print(cls)
    baseline = ap_values[cls][0]
    # print("Baseline mean:",np.mean(baseline), ", Baseline stdd:",np.std(baseline))
    for i, samples in enumerate(ap_values[cls]):
        if i == 0:
            continue
        # print("mean:",np.mean(samples), ", stdd:",np.std(samples))

        result = scipy.stats.ttest_ind(baseline, samples, equal_var=False)
        print(result)